In [1]:
# ============================================
# WEEK 6 - SQL ANALYSIS
# ============================================

import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Connect to SQLite database
conn = sqlite3.connect('../data/smartchurn.db')
cursor = conn.cursor()

print("=" * 50)
print("SQL ANALYSIS - SMARTCHURN AI")
print("=" * 50)
print("✅ Database connected successfully!")
print(f"📁 Database: smartchurn.db")

SQL ANALYSIS - SMARTCHURN AI
✅ Database connected successfully!
📁 Database: smartchurn.db


In [2]:
# ============================================
# LOAD DATA INTO SQL DATABASE
# ============================================

print("=" * 50)
print("LOADING DATA INTO DATABASE")
print("=" * 50)

# Load clean dataset
df = pd.read_csv('../data/churn_clean.csv')

# Load into SQL table
df.to_sql(
    name='customers',
    con=conn,
    if_exists='replace',
    index=False
)

# Verify data loaded
cursor.execute("SELECT COUNT(*) FROM customers")
count = cursor.fetchone()[0]

print(f"✅ Data loaded successfully!")
print(f"📊 Total customers in database: {count}")
print(f"📋 Total columns: {len(df.columns)}")
print(f"\nColumns in database:")
print("-" * 50)

cursor.execute("PRAGMA table_info(customers)")
columns = cursor.fetchall()
for col in columns:
    print(f"  {col[1]:25} | {col[2]}")

LOADING DATA INTO DATABASE
✅ Data loaded successfully!
📊 Total customers in database: 7043
📋 Total columns: 27

Columns in database:
--------------------------------------------------
  customerID                | TEXT
  gender                    | TEXT
  SeniorCitizen             | INTEGER
  Partner                   | TEXT
  Dependents                | TEXT
  tenure                    | INTEGER
  PhoneService              | TEXT
  MultipleLines             | TEXT
  InternetService           | TEXT
  OnlineSecurity            | TEXT
  OnlineBackup              | TEXT
  DeviceProtection          | TEXT
  TechSupport               | TEXT
  StreamingTV               | TEXT
  StreamingMovies           | TEXT
  Contract                  | TEXT
  PaperlessBilling          | TEXT
  PaymentMethod             | TEXT
  MonthlyCharges            | REAL
  TotalCharges              | REAL
  Churn                     | TEXT
  Churn_num                 | INTEGER
  tenure_group              | TEXT
  

In [3]:
# ============================================
# BASIC CHURN ANALYSIS QUERIES
# ============================================

print("=" * 50)
print("BASIC CHURN ANALYSIS")
print("=" * 50)

# Query 1 - Overall Churn Rate
query1 = """
SELECT 
    Churn,
    COUNT(*) as Total_Customers,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM customers), 2) as Percentage
FROM customers
GROUP BY Churn
ORDER BY Churn DESC
"""

df_churn = pd.read_sql_query(query1, conn)
print("\n📊 Query 1: Overall Churn Distribution")
print("-" * 50)
print(df_churn.to_string(index=False))

# Query 2 - Churn by Contract Type
query2 = """
SELECT 
    Contract,
    COUNT(*) as Total_Customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) as Churned,
    ROUND(SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as Churn_Rate
FROM customers
GROUP BY Contract
ORDER BY Churn_Rate DESC
"""

df_contract = pd.read_sql_query(query2, conn)
print("\n📊 Query 2: Churn by Contract Type")
print("-" * 50)
print(df_contract.to_string(index=False))

# Query 3 - Churn by Internet Service
query3 = """
SELECT 
    InternetService,
    COUNT(*) as Total_Customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) as Churned,
    ROUND(SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as Churn_Rate
FROM customers
GROUP BY InternetService
ORDER BY Churn_Rate DESC
"""

df_internet = pd.read_sql_query(query3, conn)
print("\n📊 Query 3: Churn by Internet Service")
print("-" * 50)
print(df_internet.to_string(index=False))

print("\n✅ Basic analysis complete!")

BASIC CHURN ANALYSIS

📊 Query 1: Overall Churn Distribution
--------------------------------------------------
Churn  Total_Customers  Percentage
  Yes             1869       26.54
   No             5174       73.46

📊 Query 2: Churn by Contract Type
--------------------------------------------------
      Contract  Total_Customers  Churned  Churn_Rate
Month-to-month             3875     1655       42.71
      One year             1473      166       11.27
      Two year             1695       48        2.83

📊 Query 3: Churn by Internet Service
--------------------------------------------------
InternetService  Total_Customers  Churned  Churn_Rate
    Fiber optic             3096     1297       41.89
            DSL             2421      459       18.96
             No             1526      113        7.40

✅ Basic analysis complete!
